In [30]:
import pandas as pd
from epo.tipdata.patstat import PatstatClient

In [31]:
patstat = PatstatClient()
db = patstat.orm()

This client instance is currently configured to use a test dataset with reduced number of publications (~10K).
Use PatstatClient(env='PROD') to use the complete PATSTAT dataset (>140M publications).
Use PatstatClient(env='TEST') to use the test dataset and avoid displaying this warning



In [32]:
rows = patstat.sql_query(
    "SELECT COUNT(*) AS n_applications FROM tls201_appln",
    use_legacy_sql=False,
)
rows


[{'n_applications': 323708}]

In [33]:
rows = patstat.sql_query(
    """
    SELECT appln_id, appln_auth, appln_nr, appln_filing_date, appln_filing_year, granted
    FROM tls201_appln
    LIMIT 5
    """,
    use_legacy_sql=False,
)
pd.DataFrame(rows)


,appln_id,appln_auth,appln_nr,appln_filing_date,appln_filing_year,granted
0,23307078,GB,1020555D,9999-12-31,9999,N
1,23307812,GB,1050088D,9999-12-31,9999,N
2,47805262,US,1193832D,9999-12-31,9999,Y
3,3150828,BE,527928D,9999-12-31,9999,N
4,3084616,BE,459606D,9999-12-31,9999,N


In [34]:
rows = patstat.sql_query(
    """
    SELECT appln_id, appln_auth, appln_nr, appln_filing_date, granted
    FROM tls201_appln
    WHERE appln_filing_year = 2010
      AND appln_auth = 'EP'
      AND granted = 'Y'
    """,
    use_legacy_sql=False,
)
df = pd.DataFrame(rows)
print(len(df), "applications")
df.head()


520 applications


,appln_id,appln_auth,appln_nr,appln_filing_date,granted
0,274222610,EP,10000313,2010-01-14,Y
1,274369023,EP,10000849,2010-01-28,Y
2,274681480,EP,10001469,2010-02-12,Y
3,274720647,EP,10001552,2010-02-16,Y
4,274875659,EP,10002051,2010-03-01,Y


In [35]:
rows = patstat.sql_query(
    """
    SELECT a.appln_id, a.appln_auth, a.appln_filing_year, t.appln_title
    FROM tls201_appln AS a
    JOIN tls202_appln_title AS t ON a.appln_id = t.appln_id
    WHERE a.appln_auth = 'EP'
    LIMIT 10
    """,
    use_legacy_sql=False,
)
pd.DataFrame(rows)


,appln_id,appln_auth,appln_filing_year,appln_title
0,16427397,EP,1978,A SELF-ALIGNING TWO ROW THRUST ROLLER BEARING
1,16427726,EP,1978,Aerodynamically self-governed wind turbine.
2,16430987,EP,1979,CONIC TUBE OF HELICALLY WOUND STRIP MATERIAL
3,16431185,EP,1979,THERMAL ENERGY STORAGE DEVICE
4,16431627,EP,1979,PACKAGING DEVICE AND PROCESS THEREFOR
5,16432013,EP,1979,POWER OPERATED TONGS FOR MOVING STEEL SLABS OR INGOTS
6,16432859,EP,1979,APPARATUS FOR THE MEASUREMENT OF ACCELERATIONS OF A VIBRATING BODY
7,16434050,EP,1979,WIND MOTOR
8,16434245,EP,1979,SINGLE-BLADED WIND-TURBINE ROTOR AND A METHOD FOR STARTING AND STOPPING SAME
9,16434392,EP,1979,DOOR OR WINDOW


In [36]:
rows = patstat.sql_query(
    """
    SELECT a.appln_id, a.appln_auth, a.appln_filing_year, c.cpc_class_symbol
    FROM tls201_appln AS a
    JOIN tls224_appln_cpc AS c ON a.appln_id = c.appln_id
    WHERE c.cpc_class_symbol LIKE 'G06N  10/%'
    ORDER BY a.appln_id
    """,
    use_legacy_sql=False,
)
df_quantum = pd.DataFrame(rows)
print(len(df_quantum), "rows,", df_quantum["appln_id"].nunique(), "different applications")
df_quantum.head(10)


71 rows, 40 different applications


,appln_id,appln_auth,appln_filing_year,cpc_class_symbol
0,49529356,US,2005,G06N 10/00
1,52379660,US,2003,G06N 10/00
2,486002014,US,2017,G06N 10/00
3,486106225,WO,2017,G06N 10/00
4,498096159,US,2017,G06N 10/00
5,498096165,US,2017,G06N 10/00
6,506522128,GB,2017,G06N 10/00
7,514424928,JP,2017,G06N 10/00
8,530087971,CN,2019,G06N 10/00
9,531999521,CN,2020,G06N 10/00


In [37]:
rows = patstat.sql_query(
    """
    SELECT p.psn_name,
           p.person_ctry_code,
           COUNT(DISTINCT a.appln_id) AS n_applications
    FROM tls201_appln AS a
    JOIN tls224_appln_cpc  AS c  ON a.appln_id   = c.appln_id
    JOIN tls207_pers_appln AS pa ON a.appln_id   = pa.appln_id
    JOIN tls206_person     AS p  ON pa.person_id = p.person_id
    WHERE c.cpc_class_symbol LIKE 'G06N  10/%'
      AND pa.applt_seq_nr > 0
    GROUP BY p.psn_name, p.person_ctry_code
    ORDER BY n_applications DESC
    """,
    use_legacy_sql=False,
)
pd.DataFrame(rows)


,psn_name,person_ctry_code,n_applications
0,Quantum Technologies GmbH,DE,13
1,WELLS FARGO BANK,US,3
2,1QB INFORMATION TECHNOLOGIES,CA,3
3,"STRONG FORCE EE PORTFOLIO 2022, LLC",US,2
4,SPICEWARE COMPANY,KR,2
5,GUANGXI UNIVERSITY,,2
6,IBM (INTERNATIONAL BUSINESS MACHINES CORPORATION),US,2
7,GILBERT MATTHEW J.,US,1
8,TRINA POWER CLOUD ENERGY INTERNET TECHNOLOGY COMPANY,,1
9,HITACHI,JP,1


In [38]:
from sqlalchemy import func
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN, TLS224_APPLN_CPC, TLS206_PERSON, TLS207_PERS_APPLN,
)


In [39]:
n_apps = func.count(TLS201_APPLN.appln_id.distinct())

q = (
    db.query(TLS206_PERSON.psn_name, TLS206_PERSON.person_ctry_code, n_apps.label("n_applications"))
    .select_from(TLS201_APPLN)
    .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .filter(
        TLS224_APPLN_CPC.cpc_class_symbol.like("G06N  10/%"),
        TLS207_PERS_APPLN.applt_seq_nr > 0,
    )
    .group_by(TLS206_PERSON.psn_name, TLS206_PERSON.person_ctry_code)
    .order_by(n_apps.desc())
)

patstat.df(q)


,psn_name,person_ctry_code,n_applications
0,Quantum Technologies GmbH,DE,13
1,WELLS FARGO BANK,US,3
2,1QB INFORMATION TECHNOLOGIES,CA,3
3,"STRONG FORCE EE PORTFOLIO 2022, LLC",US,2
4,SPICEWARE COMPANY,KR,2
5,GUANGXI UNIVERSITY,,2
6,IBM (INTERNATIONAL BUSINESS MACHINES CORPORATION),US,2
7,GILBERT MATTHEW J.,US,1
8,TRINA POWER CLOUD ENERGY INTERNET TECHNOLOGY COMPANY,,1
9,HITACHI,JP,1


In [40]:
from sqlalchemy import or_
from epo.tipdata.patstat.database.models import (
    TLS202_APPLN_TITLE, TLS203_APPLN_ABSTR, TLS209_APPLN_IPC,
)


In [41]:
# Words searched in titles and abstracts (lowercase; "qubit" also matches "qubits")
QC_KEYWORDS = (
    "quantum comput",          # quantum computer / computing / computation
    "qubit",
    "quantum processor",
    "quantum algorithm",
    "quantum circuit",
    "quantum gate",
    "quantum annealing",
    "quantum annealer",
    "quantum error correction",
)

# CPC codes, written WITHOUT spaces (we remove the spaces from PATSTAT's codes too)
QC_CPC_CODES = (
    "G06N10/",       # quantum computing
    "H04L9/0852",    # quantum cryptography (quantum key distribution)
    "H04L9/0855",
    "H04L9/0858",
    "H04B10/70",     # quantum communication
)

# IPC codes
QC_IPC_CODES = (
    "G06N10/",
)


In [42]:
def keyword_filter(terms):
    """True if any term is in the title or the abstract (lowercased)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *[title.like(f"%{t}%") for t in terms],
        *[abstract.like(f"%{t}%") for t in terms],
    )


def code_filter(column, codes):
    """True if the code (spaces removed) starts with one of the codes."""
    clean = func.replace(column, " ", "")
    return or_(*[clean.like(f"{c}%") for c in codes])


appln_id = TLS201_APPLN.appln_id.label("appln_id")

q_kw = (
    db.query(appln_id)
    .select_from(TLS201_APPLN)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(keyword_filter(QC_KEYWORDS))
)

q_cpc = (
    db.query(appln_id)
    .select_from(TLS201_APPLN)
    .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .filter(code_filter(TLS224_APPLN_CPC.cpc_class_symbol, QC_CPC_CODES))
)

q_ipc = (
    db.query(appln_id)
    .select_from(TLS201_APPLN)
    .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
    .filter(code_filter(TLS209_APPLN_IPC.ipc_class_symbol, QC_IPC_CODES))
)

ids_kw = set(patstat.df(q_kw.distinct())["appln_id"])
ids_cpc = set(patstat.df(q_cpc.distinct())["appln_id"])
ids_ipc = set(patstat.df(q_ipc.distinct())["appln_id"])

print("keywords:", len(ids_kw))
print("CPC:     ", len(ids_cpc))
print("IPC:     ", len(ids_ipc))


keywords: 7
CPC:      45
IPC:      27


In [43]:
ids_all = ids_kw | ids_cpc | ids_ipc        # | means "union" for Python sets

print("quantum dataset:", len(ids_all), "applications")
print("found ONLY by keywords:", len(ids_kw - ids_cpc - ids_ipc))
print("found ONLY by codes:   ", len((ids_cpc | ids_ipc) - ids_kw))

q_details = (
    db.query(TLS201_APPLN.appln_id, TLS201_APPLN.appln_auth,
             TLS201_APPLN.appln_filing_year, TLS202_APPLN_TITLE.appln_title)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .filter(TLS201_APPLN.appln_id.in_(ids_all))
)
df_qc = patstat.df(q_details)
df_qc.head(10)


quantum dataset: 48 applications
found ONLY by keywords: 1
found ONLY by codes:    41


,appln_id,appln_auth,appln_filing_year,appln_title
0,52379660,US,2003,Semiconductor quantum cryptographic device and method
1,49529356,US,2005,Method and device for performing a quantum algorithm to simulate a genetic algorithm
2,406811995,US,2010,"Communication terminal, communication system, communication method and communication program"
3,440367647,CN,2014,Optimizing method of mixed energy accumulation capacity configuration for stabilization wind power fluctuation
4,486002014,US,2017,Method and system for improving a policy for a stochastic control problem
5,506522128,GB,2017,Method and system for improving a policy for a stochastic control problem
6,514424928,JP,2017,確率制御問題に対する方策を改善する方法及びシステム
7,486106225,WO,2017,METHOD AND SYSTEM FOR IMPROVING A POLICY FOR A STOCHASTIC CONTROL PROBLEM
8,498096165,US,2017,Reducing curtailment of wind power generation
9,498096159,US,2017,REDUCING CURTAILMENT OF WIND POWER GENERATION


In [44]:
df_codes = patstat.df(
    db.query(
        TLS224_APPLN_CPC.appln_id,
        func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "").label("code"),
    )
    .filter(code_filter(TLS224_APPLN_CPC.cpc_class_symbol, QC_CPC_CODES))
)

for c in QC_CPC_CODES:
    n = df_codes.loc[df_codes["code"].str.startswith(c), "appln_id"].nunique()
    print(f"{c:12} {n} applications")


G06N10/      40 applications
H04L9/0852   10 applications
H04L9/0855   0 applications
H04L9/0858   2 applications
H04B10/70    0 applications


In [45]:
df_qc["found_by"] = df_qc["appln_id"].map(
    lambda i: "+".join(
        name for name, ids in [("kw", ids_kw), ("cpc", ids_cpc), ("ipc", ids_ipc)] if i in ids
    )
)

df_qc["found_by"].value_counts()


found_by
cpc+ipc       24
cpc           15
kw+cpc         5
ipc            2
kw             1
kw+cpc+ipc     1
Name: count, dtype: int64

In [46]:
pd.set_option("display.max_colwidth", 120)      # show long titles in full
df_qc.sort_values("found_by")[["found_by", "appln_filing_year", "appln_title"]]


,found_by,appln_filing_year,appln_title
19,cpc,2022,Systems and methods for post-quantum cryptography optimization
14,cpc,2022,Systems and methods for post-quantum cryptography optimization
18,cpc,2020,Adaptive power supply
24,cpc,2023,Policy and Governance Engines for Energy and Power Management of Edge Computing Devices
11,cpc,2019,Systems and methods for post-quantum cryptography optimization
10,cpc,2019,Systems and methods for post-quantum cryptography optimization
9,cpc,2017,REDUCING CURTAILMENT OF WIND POWER GENERATION
17,cpc,2020,Systems and methods for post-quantum cryptography on a smartcard
31,cpc,2022,"AI-BASED ENERGY EDGE PLATFORM, SYSTEMS, AND METHODS"
30,cpc,2022,"AI-BASED ENERGY EDGE PLATFORM, SYSTEMS, AND METHODS"


In [48]:
noise_rows = [20, 26, 24, 17, 14, 10, 6, 46, 23, 4, 2, 5]
noise_ids = set(df_qc.loc[noise_rows, "appln_id"])

df_ipc_codes = patstat.df(
    db.query(
        TLS209_APPLN_IPC.appln_id,
        func.replace(TLS209_APPLN_IPC.ipc_class_symbol, " ", "").label("code"),
    )
    .filter(code_filter(TLS209_APPLN_IPC.ipc_class_symbol, QC_IPC_CODES))
)

print("CPC codes of the noise:")
print(df_codes[df_codes["appln_id"].isin(noise_ids)]["code"].value_counts())
print("\nIPC codes of the noise:")
print(df_ipc_codes[df_ipc_codes["appln_id"].isin(noise_ids)]["code"].value_counts())


CPC codes of the noise:
code
G06N10/00     7
H04L9/0852    4
G06N10/60     3
G06N10/20     2
G06N10/80     2
Name: count, dtype: int64

IPC codes of the noise:
code
G06N10/00    3
G06N10/60    1
G06N10/20    1
Name: count, dtype: int64


In [49]:
# 1. Get title + abstract of every candidate patent
ids_codes = ids_cpc | ids_ipc
candidates = ids_kw | ids_codes

df_final = patstat.df(
    db.query(TLS201_APPLN.appln_id, TLS201_APPLN.appln_auth, TLS201_APPLN.appln_filing_year,
             TLS202_APPLN_TITLE.appln_title, TLS203_APPLN_ABSTR.appln_abstract)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(TLS201_APPLN.appln_id.in_(candidates))
)

# 2. Rule: keep if keywords found it, OR (a code found it AND the text mentions quantum/qubit)
text = (df_final["appln_title"].fillna("") + " " + df_final["appln_abstract"].fillna("")).str.lower()
mentions_quantum = text.str.contains("quantum|qubit")
found_by_kw = df_final["appln_id"].isin(ids_kw)
found_by_code = df_final["appln_id"].isin(ids_codes)

df_final = df_final[found_by_kw | (found_by_code & mentions_quantum)]

# 3. Look and save
print("candidates:", len(candidates), "→ final dataset:", df_final["appln_id"].nunique())
df_final.to_csv("quantum_TEST.csv", index=False)
df_final[["appln_id", "appln_filing_year", "appln_title"]].head(20)


candidates: 48 → final dataset: 21


,appln_id,appln_filing_year,appln_title
0,52379660,2003,Semiconductor quantum cryptographic device and method
1,49529356,2005,Method and device for performing a quantum algorithm to simulate a genetic algorithm
3,440367647,2014,Optimizing method of mixed energy accumulation capacity configuration for stabilization wind power fluctuation
10,590457932,2019,Systems and methods for post-quantum cryptography optimization
11,581319174,2019,Systems and methods for post-quantum cryptography optimization
13,597335947,2021,Systems and methods for post-quantum cryptography optimization
14,598413924,2022,Systems and methods for post-quantum cryptography optimization
15,530087971,2019,Optical power prediction method based on adaptive classification strategy and hybrid optimization SVR
16,602985974,2020,Systems and methods for post-quantum cryptography optimization
17,584921132,2020,Systems and methods for post-quantum cryptography on a smartcard
